In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from transformers import AutoImageProcessor, AutoModelForImageClassification
from tqdm.auto import tqdm
from torchvision import transforms

#sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_DIR = Path("geo_dataset")  # change this
TRAIN_DIR = DATA_DIR / "train"
HOLDOUT_DIR = DATA_DIR / "holdout_public"
LABELS_PATH = DATA_DIR / "train_labels.csv"

/home/utn/poli22wo/miniconda3/envs/dl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv(LABELS_PATH)

print(df.shape)
display(df.head())

(11758, 5)


,filename,country,iso,lat,lng
0,1fcb4a43864244259b7d8f4a00f1e475.jpg,Turkey,TR,40.112290,38.304629
1,742f45b0211c44ffb19ad84931ea519c.jpg,France,FR,48.094103,-1.994316
2,152a13ef249d4efa95c51ed93f026284.jpg,Turkey,TR,41.324741,27.961821
3,81ce4a88bff14fef8420bca42019b12b.jpg,France,FR,47.585855,-2.971004
4,6fbcfe523e1349759e6060d632d52e54.jpg,United_Kingdom,GB,55.698094,-4.305315


Validation Split

In [4]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["country"],
)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))

Training images: 9406
Validation images: 2352


Model Verification

In [19]:
MODEL_NAME = "apple/mobilevitv2-1.0-imagenet1k-256"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Device:", device)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 269/269 [00:00<00:00, 57127.48it/s]
[transformers] MobileViTV2ForImageClassification LOAD REPORT from: apple/mobilevitv2-1.0-imagenet1k-256
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 512]) vs model:torch.Size([2, 512])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Device: cuda


In [20]:
total_params = sum(p.numel() for p in model.parameters())

print(f"Parameters: {total_params:,}")
assert total_params <= 5_000_000

Parameters: 4,389,867


Image Processor and Dataset

In [21]:
horizontal_flip = transforms.RandomHorizontalFlip(
    p=0.5
)

In [22]:
class GeolocationDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        processor,
        transform=None,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = self.image_dir / row["filename"]
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        pixel_values = self.processor(
            images=image,
            return_tensors="pt",
        )["pixel_values"].squeeze(0)

        coordinates = torch.tensor(
            [
                row["lat"] / 90,
                row["lng"] / 180,
            ],
            dtype=torch.float32,
        )

        return pixel_values, coordinates

In [23]:
train_dataset = GeolocationDataset(
    train_df,
    TRAIN_DIR,
    processor,
    transform=horizontal_flip,
)

val_dataset = GeolocationDataset(
    val_df,
    TRAIN_DIR,
    processor,
    transform=None,
)

In [24]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

#Loss and Optimizer

In [25]:
loss_function = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
)

In [26]:
images, coordinates = next(iter(train_loader))

images = images.to(device)
coordinates = coordinates.to(device)

print("Images:", images.shape)
print("Coordinates:", coordinates.shape)

Images: torch.Size([32, 3, 256, 256])
Coordinates: torch.Size([32, 2])


In [27]:
def haversine_km(lat1, lng1, lat2, lng2):
    radius = 6371.0088

    lat1 = np.radians(lat1)
    lng1 = np.radians(lng1)
    lat2 = np.radians(lat2)
    lng2 = np.radians(lng2)

    difference = (
        np.sin((lat2 - lat1) / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin((lng2 - lng1) / 2) ** 2
    )

    return (
        2
        * radius
        * np.arcsin(
            np.sqrt(np.clip(difference, 0, 1))
        )
    )

Load Best Weights

In [28]:
saved_weights = torch.load(
    "best_model.pt",
    map_location=device,
    weights_only=True,
)

model.load_state_dict(saved_weights)

<All keys matched successfully>

In [41]:
checkpoint = torch.load(
    "training_checkpoint.pt",
    map_location=device,
    weights_only=False,
)

In [42]:
model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

start_epoch = checkpoint["epoch"]
history = checkpoint["history"]
best_median = checkpoint["best_median"]
best_epoch = checkpoint["best_epoch"]

patience = checkpoint["patience"]

In [43]:
print("Last completed epoch:", start_epoch)
print("Best epoch:", best_epoch)
print("Best median:", best_median)
print("History entries:", len(history))

Last completed epoch: 30
Best epoch: 30
Best median: 650.746
History entries: 11


Full Train + Validation Loop (Current best-> Epochs: 20, median: 709) (Reload Optimizer before continuing training)

In [ ]:
best_median = float("inf")
#best_epoch = 19

epochs_without_improvement = 0
patience = 4

In [28]:
start_epoch = 0
END_EPOCH = 40

history = []

best_median = float("inf")
best_epoch = 0

epochs_without_improvement = 0
patience = 4

best_model_path = "best_model_flip.pt"
checkpoint_path = "training_checkpoint_flip.pt"

for epoch in range(start_epoch, END_EPOCH):

    # --------------------
    # Training
    # --------------------
    model.train()
    total_training_loss = 0

    training_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Training",
    )

    for images, coordinates in training_bar:
        images = images.to(device)
        coordinates = coordinates.to(device)

        optimizer.zero_grad()

        predictions = torch.tanh(
            model(pixel_values=images).logits
        )

        loss = loss_function(
            predictions,
            coordinates,
        )

        loss.backward()
        optimizer.step()

        total_training_loss += loss.item()

        training_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_training_loss = (
        total_training_loss / len(train_loader)
    )

    # --------------------
    # Validation
    # --------------------
    model.eval()

    total_validation_loss = 0
    all_predictions = []
    all_coordinates = []

    validation_bar = tqdm(
        val_loader,
        desc=f"Epoch {epoch + 1}/{END_EPOCH} - Validation",
    )

    with torch.no_grad():
        for images, coordinates in validation_bar:
            images = images.to(device)
            coordinates = coordinates.to(device)

            predictions = torch.tanh(
                model(pixel_values=images).logits
            )

            loss = loss_function(
                predictions,
                coordinates,
            )

            total_validation_loss += loss.item()

            all_predictions.append(
                predictions.cpu().numpy()
            )

            all_coordinates.append(
                coordinates.cpu().numpy()
            )

    average_validation_loss = (
        total_validation_loss / len(val_loader)
    )

    # Combine validation batches
    all_predictions = np.concatenate(all_predictions)
    all_coordinates = np.concatenate(all_coordinates)

    # Convert normalized coordinates back into degrees
    predictions_degrees = all_predictions.copy()
    coordinates_degrees = all_coordinates.copy()

    predictions_degrees[:, 0] *= 90
    predictions_degrees[:, 1] *= 180

    coordinates_degrees[:, 0] *= 90
    coordinates_degrees[:, 1] *= 180

    # Calculate geographic distances
    distances = haversine_km(
        coordinates_degrees[:, 0],
        coordinates_degrees[:, 1],
        predictions_degrees[:, 0],
        predictions_degrees[:, 1],
    )

    mean_distance = np.mean(distances)
    median_distance = np.median(distances)
    within_200 = np.mean(distances < 200)
    within_750 = np.mean(distances < 750)

    # Save this epoch's results
    history.append({
        "epoch": epoch + 1,
        "training_loss": average_training_loss,
        "validation_loss": average_validation_loss,
        "mean_km": mean_distance,
        "median_km": median_distance,
        "within_200": within_200,
        "within_750": within_750,
    })

    # Display this epoch's results
    print(f"\nEpoch {epoch + 1} results")
    print(f"Training loss: {average_training_loss:.4f}")
    print(f"Validation loss: {average_validation_loss:.4f}")
    print(f"Mean distance: {mean_distance:.1f} km")
    print(f"Median distance: {median_distance:.1f} km")
    print(f"Within 200 km: {within_200:.2%}")
    print(f"Within 750 km: {within_750:.2%}")

    if median_distance < best_median:
        best_median = median_distance
        best_epoch = epoch + 1

        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            best_model_path,
        )

        print("Saved new best model.")

    else:
        epochs_without_improvement += 1

        print(
            "Epochs without improvement:",
            epochs_without_improvement,
        )

    if epochs_without_improvement >= patience:
        print("Early stopping.")
        break

Epoch 1/40 - Validation: 100%|██████████| 74/74 [00:16<00:00,  4.51it/s]



Epoch 1 results
Training loss: 0.0753
Validation loss: 0.0098
Mean distance: 1406.1 km
Median distance: 1355.2 km
Within 200 km: 1.40%
Within 750 km: 17.05%
Saved new best model.


Epoch 2/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.26it/s]



Epoch 2 results
Training loss: 0.0054
Validation loss: 0.0052
Mean distance: 985.7 km
Median distance: 858.3 km
Within 200 km: 4.04%
Within 750 km: 42.35%
Saved new best model.


Epoch 3/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.41it/s]


...


Epoch 27 results
Training loss: 0.0006
Validation loss: 0.0033
Mean distance: 747.8 km
Median distance: 622.2 km
Within 200 km: 10.63%
Within 750 km: 60.08%
Epochs without improvement: 2


Epoch 28/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.19it/s]



Epoch 28 results
Training loss: 0.0005
Validation loss: 0.0033
Mean distance: 767.2 km
Median distance: 647.6 km
Within 200 km: 9.61%
Within 750 km: 58.08%
Epochs without improvement: 3


Epoch 29/40 - Validation: 100%|██████████| 74/74 [00:11<00:00,  6.51it/s]


Epoch 29 results
Training loss: 0.0006
Validation loss: 0.0032
Mean distance: 749.8 km
Median distance: 629.2 km
Within 200 km: 9.95%
Within 750 km: 59.27%
Epochs without improvement: 4
Early stopping.


In [29]:
torch.save(
    {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_median": best_median,
        "history": history,
        "best_epoch": best_epoch,
        "patience": patience,
        "epochs_without_improvement": epochs_without_improvement
    },
    "training_checkpoint_flip.pt",
)

In [46]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    "baseline_history_40_epochs.csv",
    index=False,
)